In [ ]:
!pip install easyocr pandas pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 125.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())


GPU available: True


In [ ]:
import requests
import easyocr
from io import BytesIO
from PIL import Image
import pandas as pd
import re
import os

# Initialize EasyOCR reader
reader = easyocr.Reader(['en'])

# Afghanistan image pages
base_url = "https://reader.scottonline.com/sample/afghanistan/files/pages/tablet/"
page_range = range(4, 19)

all_results = []

print("🔍 Starting OCR extraction...")
for i in page_range:
    url = f"{base_url}{i}.jpg"
    print(f"🔍 Processing page {i}: {url}")
    try:
        response = requests.get(url)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))

        # OCR text
        results = reader.readtext(img, detail=0)
        all_results.extend([line.strip() for line in results if line.strip() != ""])
    except Exception as e:
        print(f"❌ Failed to process page {i}: {e}")

# Group into chunks by Scott number
chunks = []
current_chunk = []

for line in all_results:
    if re.match(r"^\d{3,4}[A-Z]?$", line) and not re.match(r"^19\d{2}$", line):
        if current_chunk:
            chunks.append(current_chunk)
        current_chunk = [line]
    else:
        current_chunk.append(line)

if current_chunk:
    chunks.append(current_chunk)

# Optional debug
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i} ---")
    print("\n".join(chunk))
    print()

# Define the parser
def parse_chunk(chunk):
    text = " ".join(chunk).replace(",", ".")
    scott_match = re.search(r"\b(\d{3,4}[A-Z]?)\b", text)
    year_match = re.search(r"\b(19\d{2})\b", text)
    prices = re.findall(r"\d+\.\d{2}", text)
    desc_color_match = re.search(
        r"\b\d{3,4}[A-Z]?\b\s+(.*?)([A-Za-z]{3,}.*?)?(?=\d{4}|\$|\d+\.\d{2}|$)",
        text
    )

    return {
        "Scott Number": scott_match.group(1) if scott_match else None,
        "Description": desc_color_match.group(1).strip() if desc_color_match else "",
        "Color": desc_color_match.group(2).strip() if desc_color_match and desc_color_match.lastindex >= 2 else "",
        "Year": year_match.group(1) if year_match else None,
        "Mint Price": prices[0] if len(prices) > 0 else None,
        "Used Price": prices[1] if len(prices) > 1 else None
    }

# Parse chunks
parsed_data = []
for chunk in chunks:
    parsed = parse_chunk(chunk)
    if parsed["Scott Number"]:
        parsed_data.append(parsed)

# Convert to DataFrame
df = pd.DataFrame(parsed_data)
df.head()


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete🔍 Starting OCR extraction...
🔍 Processing page 4: https://reader.scottonline.com/sample/afghanistan/files/pages/tablet/4.jpg
🔍 Processing page 5: https://reader.scottonline.com/sample/afghanistan/files/pages/tablet/5.jpg
🔍 Processing page 6: https://reader.scottonline.com/sample/afghanistan/files/pages/tablet/6.jpg
🔍 Processing page 7: https://reader.scottonline.com/sample/afghanistan/files/pages/tablet/7.jpg
🔍 Processing page 8: https://reader.scottonline.com/sample/afghanistan/files/pages/tablet/8.jpg
🔍 Processing page 9: https://reader.scottonline.com/sample/afghanistan/files/pages/tablet/9.jpg
🔍 Processing page 10: https://reader.scottonline.com/sample/afghanistan/files/pages/tablet/10.jpg
🔍 Processing page 11: https://reader.scottonline.com/sample/afghanistan/files/pages/tablet/11.jpg


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import requests
import easyocr
from io import BytesIO
from PIL import Image
import pandas as pd
import re
import os
import time
from google.colab import files
import torch

# Confirm GPU is available
print("GPU available:", torch.cuda.is_available())

# Initialize EasyOCR with GPU
reader = easyocr.Reader(['en'], gpu=True)

# Country list (no spaces in URLs)
countries = [
    "argentina", "austria", "belgium", "bulgaria", "canada",
    "chile", "colombia", "cuba", "france", "germany", "greatbritain",
    "greece", "hungary", "india", "jamaica", "japan", "korea", "malaysia",
    "mexico", "netherlands", "newzealand", "pakistan", "phillippines",
    "poland", "saudiarabia", "southafrica", "spain", "thailand", "turkey"
]

# Create folder for debug
os.makedirs("debug_chunks", exist_ok=True)

# All data collected
all_countries_data = []

def parse_chunk(chunk):
    text = " ".join(chunk).replace(",", ".")
    scott_match = re.search(r"\b(\d{3,4}[A-Z]?)\b", text)
    year_match = re.search(r"\b(19\d{2})\b", text)
    prices = re.findall(r"\d+\.\d{2}", text)
    desc_color_match = re.search(
        r"\b\d{3,4}[A-Z]?\b\s+(.*?)([A-Za-z]{3,}.*?)?(?=\d{4}|\$|\d+\.\d{2}|$)",
        text
    )
    return {
        "Scott Number": scott_match.group(1) if scott_match else None,
        "Description": desc_color_match.group(1).strip() if desc_color_match else "",
        "Color": desc_color_match.group(2).strip() if desc_color_match and desc_color_match.lastindex >= 2 else "",
        "Year": year_match.group(1) if year_match else None,
        "Mint Price": prices[0] if len(prices) > 0 else None,
        "Used Price": prices[1] if len(prices) > 1 else None
    }

def extract_country(country, page_range=range(1, 11)):
    base_url = f"https://reader.scottonline.com/sample/{country}/files/pages/tablet/"
    all_results = []

    print(f"🌍 Starting {country}")
    for i in page_range:
        url = f"{base_url}{i}.jpg"
        print(f"  📄 Page {i}: {url}")
        try:
            response = requests.get(url)
            response.raise_for_status()
            img = Image.open(BytesIO(response.content))
            results = reader.readtext(img, detail=0)
            all_results.extend([line.strip() for line in results if line.strip()])
        except Exception as e:
            print(f"  ❌ Failed to process page {i}: {e}")

    # Chunking lines by stamp entry
    chunks = []
    current_chunk = []

    for line in all_results:
        if re.match(r"^\d{3,4}[A-Z]?", line):
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = [line]
        else:
            current_chunk.append(line)
    if current_chunk:
        chunks.append(current_chunk)

    # Debug save
    with open(f"debug_chunks/{country}_chunks.txt", "w") as f:
        for i, chunk in enumerate(chunks):
            f.write(f"=== Chunk {i} ===\n")
            f.write("\n".join(chunk))
            f.write("\n\n")

    # Parse each chunk
    parsed_data = []
    for chunk in chunks:
        parsed = parse_chunk(chunk)
        if parsed["Scott Number"]:
            parsed["Country"] = country
            all_countries_data.append(parsed)
            parsed_data.append(parsed)

    return parsed_data

# Process and download per country
for country in countries:
    country_data = extract_country(country)
    df_country = pd.DataFrame(country_data)

    if not df_country.empty:
        filename = f"{country}_stamps.csv"
        df_country.to_csv(filename, index=False)
        files.download(filename)

    time.sleep(1)  # Be kind to the server


GPU available: True
🌍 Starting argentina
  📄 Page 1: https://reader.scottonline.com/sample/argentina/files/pages/tablet/1.jpg
  📄 Page 2: https://reader.scottonline.com/sample/argentina/files/pages/tablet/2.jpg
  📄 Page 3: https://reader.scottonline.com/sample/argentina/files/pages/tablet/3.jpg
  📄 Page 4: https://reader.scottonline.com/sample/argentina/files/pages/tablet/4.jpg
  📄 Page 5: https://reader.scottonline.com/sample/argentina/files/pages/tablet/5.jpg
  📄 Page 6: https://reader.scottonline.com/sample/argentina/files/pages/tablet/6.jpg
  📄 Page 7: https://reader.scottonline.com/sample/argentina/files/pages/tablet/7.jpg
  📄 Page 8: https://reader.scottonline.com/sample/argentina/files/pages/tablet/8.jpg
  📄 Page 9: https://reader.scottonline.com/sample/argentina/files/pages/tablet/9.jpg
  📄 Page 10: https://reader.scottonline.com/sample/argentina/files/pages/tablet/10.jpg


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🌍 Starting austria
  📄 Page 1: https://reader.scottonline.com/sample/austria/files/pages/tablet/1.jpg
  📄 Page 2: https://reader.scottonline.com/sample/austria/files/pages/tablet/2.jpg
  📄 Page 3: https://reader.scottonline.com/sample/austria/files/pages/tablet/3.jpg
  📄 Page 4: https://reader.scottonline.com/sample/austria/files/pages/tablet/4.jpg
  📄 Page 5: https://reader.scottonline.com/sample/austria/files/pages/tablet/5.jpg
  📄 Page 6: https://reader.scottonline.com/sample/austria/files/pages/tablet/6.jpg
  📄 Page 7: https://reader.scottonline.com/sample/austria/files/pages/tablet/7.jpg
  📄 Page 8: https://reader.scottonline.com/sample/austria/files/pages/tablet/8.jpg
  📄 Page 9: https://reader.scottonline.com/sample/austria/files/pages/tablet/9.jpg
  📄 Page 10: https://reader.scottonline.com/sample/austria/files/pages/tablet/10.jpg


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🌍 Starting turkey
  📄 Page 1: https://reader.scottonline.com/sample/turkey/files/pages/tablet/1.jpg
  📄 Page 2: https://reader.scottonline.com/sample/turkey/files/pages/tablet/2.jpg
  📄 Page 3: https://reader.scottonline.com/sample/turkey/files/pages/tablet/3.jpg
  📄 Page 4: https://reader.scottonline.com/sample/turkey/files/pages/tablet/4.jpg
  📄 Page 5: https://reader.scottonline.com/sample/turkey/files/pages/tablet/5.jpg
  📄 Page 6: https://reader.scottonline.com/sample/turkey/files/pages/tablet/6.jpg
  📄 Page 7: https://reader.scottonline.com/sample/turkey/files/pages/tablet/7.jpg
  📄 Page 8: https://reader.scottonline.com/sample/turkey/files/pages/tablet/8.jpg
  📄 Page 9: https://reader.scottonline.com/sample/turkey/files/pages/tablet/9.jpg
  📄 Page 10: https://reader.scottonline.com/sample/turkey/files/pages/tablet/10.jpg


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import glob
import os

# Optional: list files in the current directory
# !ls

# Find all *_stamps.csv files in the current directory
csv_files = glob.glob("*_stamps.csv")

# Read and concatenate all CSVs
merged_df = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)

# Save merged file
merged_filename = "all_parsed_stamps.csv"
merged_df.to_csv(merged_filename, index=False)

# Download the merged file
from google.colab import files
files.download(merged_filename)

print(f"✅ Merged {len(csv_files)} files into '{merged_filename}' with {len(merged_df)} rows.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Merged 28 files into 'all_parsed_stamps.csv' with 9718 rows.


Cleaning

In [ ]:
import pandas as pd
import numpy as np
import re

# Load data
df = pd.read_csv("all_parsed_stamps.csv")

# Make a copy to preserve the original
df_clean = df.copy()

# 1. Standardize column formats
df_clean['Scott Number'] = df_clean['Scott Number'].astype(str).str.strip()
df_clean['Country'] = df_clean['Country'].str.strip().str.title()

# 2. Handle missing values and clean text columns
for col in ['Description']: # Removed 'Color'
    if col in df_clean:
        df_clean[col] = df_clean[col].astype(str).str.strip()
        df_clean[col] = df_clean[col].replace({'nan': np.nan})
        df_clean[col] = df_clean[col].apply(
            lambda x: re.sub(r'\(\s*$', '', x) if isinstance(x, str) else x
        )

# Convert Year to integer where possible
df_clean['Year'] = pd.to_numeric(df_clean['Year'], errors='coerce').astype('Int64')

# Prices: ensure float, keep NaN for missing
for col in ['Mint Price', 'Used Price']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').round(2)

# Function to clean text entries
def clean_text(val):
    if not isinstance(val, str):
        return val
    val = re.sub(r'[\(\)]', '', val)  # remove parentheses
    val = re.sub(r'\s+', ' ', val).strip()  # remove extra spaces
    val = val.replace("Plennigs", "Pfennigs")  # fix known typos
    return val

df_clean['Description'] = df_clean['Description'].apply(clean_text)
# Removed 'Color' cleaning: df_clean['Color'] = df_clean['Color'].apply(clean_text)


# Drop the 'Color' column
if 'Color' in df_clean.columns:
    df_clean = df_clean.drop('Color', axis=1)

# 3. Deduplicate by Scott Number & Country
def pick_fullest(group):
    return group.loc[group.notna().sum(axis=1).idxmax()]

df_clean = df_clean.groupby(['Scott Number', 'Country'], as_index=False).apply(pick_fullest)

# 4. Remove invalid Scott Numbers
def is_valid_scott(num):
    if not isinstance(num, str):
        return False
    num_str = num.strip()
    if num_str == "":
        return False
    # All zeros
    if re.fullmatch(r"0+", num_str):
        return False
    # Keep if contains at least one digit and only letters/digits/hyphens
    if re.fullmatch(r"[A-Za-z0-9\-]+", num_str):
        return True
    return False

df_clean = df_clean[df_clean['Scott Number'].astype(str).apply(is_valid_scott)]

# 5. Optional enrichment: price ratio
df_clean['Price Ratio'] = df_clean.apply(
    lambda row: row['Mint Price'] / row['Used Price']
    if pd.notna(row['Mint Price']) and pd.notna(row['Used Price']) and row['Used Price'] != 0
    else np.nan,
    axis=1
)

# Final tidy-up
df_clean.reset_index(drop=True, inplace=True)

# Save cleaned dataset
df_clean.to_csv("cleaned_stamps.csv", index=False)

/tmp/ipython-input-2254845914.py:52: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_clean = df_clean.groupby(['Scott Number', 'Country'], as_index=False).apply(pick_fullest)


In [ ]:
display(df_clean.head())